# M4 · Sistemes de Big Data — RA1
## Notebook 2 · Pandas vs Polars · CSV vs Parquet

**Actividad de refuerzo — no evaluable**

En este notebook vamos a estudiar una idea fundamental:

> **cuando aumenta el volumen de datos, no solo importa el algoritmo: también importan la herramienta y el formato de almacenamiento.**

Trabajaremos con un dataset relativamente grande y compararemos:

- **Pandas** y **Polars**;
- **CSV** y **Parquet**;
- ejecución **eager** y **lazy**;
- lectura completa frente a lectura selectiva de columnas;
- filtrado y agregaciones;
- tiempo, tamaño de archivo y memoria.

---

### Objetivos

Al terminar deberías ser capaz de explicar:

1. por qué CSV y Parquet no almacenan los datos de la misma forma;
2. por qué Parquet suele ser especialmente útil en analítica;
3. qué diferencias conceptuales existen entre Pandas y Polars;
4. qué significa **lazy execution**;
5. qué son **projection pushdown** y **predicate pushdown**;
6. por qué un benchmark aislado no demuestra que una herramienta sea siempre mejor;
7. cómo elegir herramienta y formato según el problema.

---

> **Importante:** los tiempos obtenidos dependerán del ordenador, Colab, caché, versiones de librerías y carga del sistema.  
> No buscamos memorizar números concretos, sino observar **tendencias**.

# 1. Preparar el entorno

Instalaremos:

- `pandas`
- `polars`
- `pyarrow`

`pyarrow` permite trabajar con Apache Parquet desde Pandas.

En Google Colab normalmente Pandas ya está instalado, pero ejecutaremos la instalación para asegurar que disponemos de las dependencias necesarias.

In [ ]:
!pip -q install polars pyarrow

In [ ]:
import os
import gc
import time
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)
print("Polars:", pl.__version__)
print("NumPy:", np.__version__)

## Función para medir tiempos

Ejecutaremos cada operación varias veces y utilizaremos la **mediana**.

La mediana suele ser más estable que utilizar una única ejecución.

In [ ]:
def medir(func, repeticiones=3):
    tiempos = []

    for _ in range(repeticiones):
        gc.collect()

        inicio = time.perf_counter()
        resultado = func()
        fin = time.perf_counter()

        tiempos.append(fin - inicio)

    return statistics.median(tiempos), resultado

resultados = []

# 2. Crear un dataset de tamaño considerable

Para que la comparación tenga sentido necesitamos bastantes filas.

Generaremos un dataset sintético de transacciones de comercio electrónico con:

- timestamp;
- cliente;
- producto;
- categoría;
- país;
- canal;
- unidades;
- precio;
- descuento;
- facturación.

El dataset es sintético porque aquí queremos controlar el tamaño y poder repetir exactamente el experimento.

---

### Tamaño

Por defecto utilizaremos:

```text
750.000 filas
```

Si tu entorno funciona con facilidad, después puedes probar:

```python
N_ROWS = 1_500_000
```

o incluso más.

**No aumentes el tamaño directamente si trabajas en un equipo con poca memoria.**

In [ ]:
N_ROWS = 750_000
SEED = 42

rng = np.random.default_rng(SEED)

categories = np.array([
    "electronics",
    "home",
    "beauty",
    "sports",
    "food",
    "fashion",
    "books",
    "toys"
])

countries = np.array([
    "ES", "FR", "DE", "IT", "PT", "NL", "BE", "IE"
])

channels = np.array([
    "web", "app", "marketplace", "store"
])

base_date = np.datetime64("2026-01-01T00:00:00")
seconds_in_8_months = 60 * 60 * 24 * 240

df_base = pd.DataFrame({
    "transaction_id": np.arange(1, N_ROWS + 1),
    "timestamp": base_date + rng.integers(
        0,
        seconds_in_8_months,
        size=N_ROWS
    ).astype("timedelta64[s]"),
    "customer_id": rng.integers(1, 120_000, size=N_ROWS),
    "product_id": rng.integers(1, 8_000, size=N_ROWS),
    "category": rng.choice(categories, size=N_ROWS),
    "country": rng.choice(
        countries,
        size=N_ROWS,
        p=[0.42, 0.12, 0.11, 0.10, 0.09, 0.06, 0.05, 0.05]
    ),
    "channel": rng.choice(
        channels,
        size=N_ROWS,
        p=[0.45, 0.35, 0.12, 0.08]
    ),
    "units": rng.choice(
        [1, 2, 3, 4, 5],
        size=N_ROWS,
        p=[0.62, 0.22, 0.09, 0.05, 0.02]
    ),
    "unit_price": np.round(
        rng.lognormal(mean=3.4, sigma=0.65, size=N_ROWS),
        2
    ),
    "discount": rng.choice(
        [0.00, 0.05, 0.10, 0.15, 0.20],
        size=N_ROWS,
        p=[0.55, 0.12, 0.18, 0.10, 0.05]
    )
})

df_base["revenue"] = (
    df_base["units"] * df_base["unit_price"]
).round(2)

print(f"Filas: {len(df_base):,}")
print(f"Columnas: {df_base.shape[1]}")
df_base.head()

In [ ]:
df_base.info()

# 3. Guardar exactamente los mismos datos en CSV y Parquet

Esto es importante:

> **el contenido lógico será el mismo; lo que cambia será el formato de almacenamiento.**

Guardaremos:

- `transactions.csv`
- `transactions.parquet`

Para Parquet utilizaremos compresión `snappy`, una opción habitual porque ofrece un buen equilibrio entre velocidad y compresión.

In [ ]:
DATA_DIR = Path("benchmark_data")
DATA_DIR.mkdir(exist_ok=True)

csv_path = DATA_DIR / "transactions.csv"
parquet_path = DATA_DIR / "transactions.parquet"

df_base.to_csv(csv_path, index=False)

df_base.to_parquet(
    parquet_path,
    index=False,
    compression="snappy"
)

print("Archivos creados.")

## Comparar tamaño en disco

CSV almacena los datos fundamentalmente como **texto**.

Parquet es un formato:

- binario;
- columnar;
- tipado;
- comprimible;
- diseñado para analítica.

Veamos cuánto ocupa cada archivo.

In [ ]:
def mb(path):
    return os.path.getsize(path) / (1024 ** 2)

csv_mb = mb(csv_path)
parquet_mb = mb(parquet_path)

print(f"CSV:     {csv_mb:.2f} MB")
print(f"Parquet: {parquet_mb:.2f} MB")
print(f"Parquet ocupa el {parquet_mb / csv_mb * 100:.1f}% del tamaño del CSV.")

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(["CSV", "Parquet"], [csv_mb, parquet_mb])
plt.ylabel("Tamaño (MB)")
plt.title("Tamaño en disco")
plt.show()

### Primera conclusión

Que Parquet sea más pequeño **no es una ley universal**.

La diferencia depende de:

- tipos de datos;
- repetición de valores;
- compresión;
- número de columnas;
- contenido concreto.

Pero en datasets analíticos grandes es habitual que Parquet consiga una reducción importante.

---

# 4. Leer CSV con Pandas

CSV no almacena un esquema rico de tipos como Parquet.

Al leerlo debemos interpretar:

- números;
- strings;
- fechas.

Para que la comparación sea razonable, pediremos a Pandas que convierta `timestamp` a fecha.

In [ ]:
def pandas_read_csv():
    return pd.read_csv(
        csv_path,
        parse_dates=["timestamp"]
    )

t_pd_csv, pd_csv = medir(
    pandas_read_csv,
    repeticiones=3
)

print(f"Pandas + CSV: {t_pd_csv:.4f} s")
pd_csv.head()

# 5. Leer Parquet con Pandas

Ahora leemos exactamente los mismos datos desde Parquet.

In [ ]:
def pandas_read_parquet():
    return pd.read_parquet(parquet_path)

t_pd_parquet, pd_parquet = medir(
    pandas_read_parquet,
    repeticiones=3
)

print(f"Pandas + Parquet: {t_pd_parquet:.4f} s")
pd_parquet.head()

# 6. Leer CSV con Polars

In [ ]:
def polars_read_csv():
    return pl.read_csv(
        csv_path,
        try_parse_dates=True
    )

t_pl_csv, pl_csv = medir(
    polars_read_csv,
    repeticiones=3
)

print(f"Polars + CSV: {t_pl_csv:.4f} s")
pl_csv.head()

# 7. Leer Parquet con Polars

In [ ]:
def polars_read_parquet():
    return pl.read_parquet(parquet_path)

t_pl_parquet, pl_parquet = medir(
    polars_read_parquet,
    repeticiones=3
)

print(f"Polars + Parquet: {t_pl_parquet:.4f} s")
pl_parquet.head()

# 8. Comparación inicial de lectura

Guardaremos los tiempos en una tabla.

**No interpretes la tabla como una clasificación universal.**

En otra máquina podrían cambiar los números.

Lo importante es observar:

- si CSV y Parquet se comportan igual;
- si Pandas y Polars tienen el mismo coste;
- qué diferencias son pequeñas o grandes.

In [ ]:
lectura = pd.DataFrame({
    "herramienta": [
        "Pandas",
        "Pandas",
        "Polars",
        "Polars"
    ],
    "formato": [
        "CSV",
        "Parquet",
        "CSV",
        "Parquet"
    ],
    "tiempo_s": [
        t_pd_csv,
        t_pd_parquet,
        t_pl_csv,
        t_pl_parquet
    ]
})

lectura["tiempo_s"] = lectura["tiempo_s"].round(4)
lectura

In [ ]:
plt.figure(figsize=(8, 4))
labels = lectura["herramienta"] + " + " + lectura["formato"]

plt.bar(labels, lectura["tiempo_s"])
plt.ylabel("Tiempo mediano (s)")
plt.title("Lectura del dataset completo")
plt.xticks(rotation=20)
plt.show()

# 9. ¿Qué ocurre con los tipos?

Vamos a comparar el esquema recuperado.

Parquet almacena metadatos sobre los tipos.

CSV, en cambio, contiene principalmente texto y requiere inferencia/conversión al leerlo.

In [ ]:
print("PANDAS DESDE CSV")
print(pd_csv.dtypes)

print("\nPANDAS DESDE PARQUET")
print(pd_parquet.dtypes)

print("\nPOLARS DESDE CSV")
print(pl_csv.schema)

print("\nPOLARS DESDE PARQUET")
print(pl_parquet.schema)

### Fíjate especialmente en `timestamp`

Preguntas:

1. ¿Se ha recuperado como fecha?
2. ¿Ha sido necesario indicar algo al leer CSV?
3. ¿Qué ventaja tiene que el formato conserve el esquema?

---

# 10. Comparar memoria en RAM

El tamaño del archivo en disco y la memoria utilizada por un DataFrame **no son lo mismo**.

Vamos a obtener una estimación.

In [ ]:
pandas_memory_mb = (
    pd_parquet.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

polars_memory_mb = (
    pl_parquet.estimated_size("mb")
)

print(f"Pandas DataFrame: {pandas_memory_mb:.2f} MB")
print(f"Polars DataFrame: {polars_memory_mb:.2f} MB")

La representación interna de Pandas y Polars es diferente.

Por eso dos DataFrames con los mismos datos pueden utilizar cantidades de memoria distintas.

No significa que una librería vaya a utilizar siempre menos memoria en cualquier situación.

---

# 11. Operación 1 — Filtrado

Buscaremos transacciones que cumplan simultáneamente:

- país = `ES`;
- categoría = `electronics`;
- facturación superior a 100 €.

Primero con Pandas.

In [ ]:
def filtro_pandas():
    return pd_parquet[
        (pd_parquet["country"] == "ES")
        & (pd_parquet["category"] == "electronics")
        & (pd_parquet["revenue"] > 100)
    ]

t_filter_pd, result_filter_pd = medir(
    filtro_pandas,
    repeticiones=5
)

print(f"Pandas: {t_filter_pd:.6f} s")
print(f"Filas resultantes: {len(result_filter_pd):,}")

Ahora la misma operación con Polars.

In [ ]:
def filtro_polars():
    return pl_parquet.filter(
        (pl.col("country") == "ES")
        & (pl.col("category") == "electronics")
        & (pl.col("revenue") > 100)
    )

t_filter_pl, result_filter_pl = medir(
    filtro_polars,
    repeticiones=5
)

print(f"Polars: {t_filter_pl:.6f} s")
print(f"Filas resultantes: {result_filter_pl.height:,}")

In [ ]:
print("¿Devuelven el mismo número de filas?")
print(len(result_filter_pd) == result_filter_pl.height)

# 12. Operación 2 — Crear una variable

Calcularemos la facturación neta después del descuento:

\[
net\_revenue =
units \times unit\_price \times (1 - discount)
\]

In [ ]:
def calcular_pandas():
    temp = pd_parquet.copy()

    temp["net_revenue"] = (
        temp["units"]
        * temp["unit_price"]
        * (1 - temp["discount"])
    )

    return temp

t_calc_pd, pd_calc = medir(
    calcular_pandas,
    repeticiones=3
)

print(f"Pandas: {t_calc_pd:.5f} s")

In [ ]:
def calcular_polars():
    return pl_parquet.with_columns(
        (
            pl.col("units")
            * pl.col("unit_price")
            * (1 - pl.col("discount"))
        ).alias("net_revenue")
    )

t_calc_pl, pl_calc = medir(
    calcular_polars,
    repeticiones=3
)

print(f"Polars: {t_calc_pl:.5f} s")

# 13. Operación 3 — GroupBy / agregación

Queremos responder:

> **¿Cuánta facturación neta genera cada categoría?**

Además calcularemos:

- número de transacciones;
- unidades vendidas;
- ticket medio.

In [ ]:
def groupby_pandas():
    temp = pd_parquet.copy()

    temp["net_revenue"] = (
        temp["units"]
        * temp["unit_price"]
        * (1 - temp["discount"])
    )

    return (
        temp
        .groupby("category")
        .agg(
            transactions=("transaction_id", "count"),
            units=("units", "sum"),
            net_revenue=("net_revenue", "sum"),
            avg_ticket=("net_revenue", "mean")
        )
        .sort_values("net_revenue", ascending=False)
    )

t_group_pd, group_pd = medir(
    groupby_pandas,
    repeticiones=3
)

print(f"Pandas: {t_group_pd:.5f} s")
group_pd

In [ ]:
def groupby_polars():
    return (
        pl_parquet
        .with_columns(
            (
                pl.col("units")
                * pl.col("unit_price")
                * (1 - pl.col("discount"))
            ).alias("net_revenue")
        )
        .group_by("category")
        .agg(
            pl.len().alias("transactions"),
            pl.col("units").sum().alias("units"),
            pl.col("net_revenue").sum().alias("net_revenue"),
            pl.col("net_revenue").mean().alias("avg_ticket")
        )
        .sort("net_revenue", descending=True)
    )

t_group_pl, group_pl = medir(
    groupby_polars,
    repeticiones=3
)

print(f"Polars: {t_group_pl:.5f} s")
group_pl

# 14. Resumen de operaciones en memoria

In [ ]:
operaciones = pd.DataFrame({
    "operacion": [
        "Filtrado",
        "Nueva columna",
        "GroupBy"
    ],
    "Pandas_s": [
        t_filter_pd,
        t_calc_pd,
        t_group_pd
    ],
    "Polars_s": [
        t_filter_pl,
        t_calc_pl,
        t_group_pl
    ]
})

operaciones.round(6)

In [ ]:
x = np.arange(len(operaciones))
width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - width/2, operaciones["Pandas_s"], width, label="Pandas")
plt.bar(x + width/2, operaciones["Polars_s"], width, label="Polars")
plt.xticks(x, operaciones["operacion"])
plt.ylabel("Tiempo mediano (s)")
plt.title("Operaciones sobre datos ya cargados en memoria")
plt.legend()
plt.show()

# 15. Eager execution vs Lazy execution

Hasta ahora hemos utilizado:

```python
pl.read_parquet(...)
```

Esto carga los datos inmediatamente.

Se denomina **eager execution**:

> ejecutamos cada operación cuando la escribimos.

Polars también permite trabajar de forma **lazy**:

```python
pl.scan_parquet(...)
```

En este caso, Polars construye primero un **plan de consulta**.

No ejecuta realmente la consulta hasta que utilizamos:

```python
.collect()
```

Esto permite al motor intentar optimizar el plan antes de ejecutarlo.

In [ ]:
lazy_df = pl.scan_parquet(parquet_path)

print(type(lazy_df))

Vamos a construir una consulta:

1. leer Parquet;
2. quedarnos con España;
3. seleccionar unas pocas columnas;
4. calcular `net_revenue`;
5. agrupar por categoría;
6. ordenar el resultado.

Todavía **no se ejecutará**.

In [ ]:
query = (
    pl.scan_parquet(parquet_path)

    .filter(
        pl.col("country") == "ES"
    )

    .select([
        "category",
        "units",
        "unit_price",
        "discount"
    ])

    .with_columns(
        (
            pl.col("units")
            * pl.col("unit_price")
            * (1 - pl.col("discount"))
        ).alias("net_revenue")
    )

    .group_by("category")

    .agg(
        pl.len().alias("transactions"),
        pl.col("net_revenue").sum().alias("net_revenue")
    )

    .sort(
        "net_revenue",
        descending=True
    )
)

query

## Ver el plan optimizado

Polars puede mostrar cómo piensa ejecutar la consulta.

In [ ]:
print(query.explain(optimized=True))

En el plan pueden aparecer optimizaciones como:

### Projection pushdown

Si solo necesitamos:

```text
category
units
unit_price
discount
country
```

el motor puede evitar leer columnas innecesarias.

### Predicate pushdown

Si filtramos:

```python
country == "ES"
```

el motor intenta aplicar el filtro lo antes posible.

La idea es sencilla:

> **leer y procesar únicamente los datos necesarios.**

# 16. Ejecutar la consulta lazy

Ahora sí utilizamos:

```python
.collect()
```

In [ ]:
def ejecutar_lazy():
    return query.collect()

t_lazy, lazy_result = medir(
    ejecutar_lazy,
    repeticiones=3
)

print(f"Consulta lazy: {t_lazy:.5f} s")
lazy_result

# 17. Leer todo vs leer únicamente columnas necesarias

Parquet es especialmente interesante cuando un dataset tiene muchas columnas pero una consulta necesita solo unas pocas.

Vamos a comparar en Pandas:

### Opción A
Leer todas las columnas.

### Opción B
Leer únicamente:

- `country`
- `category`
- `revenue`

In [ ]:
columnas_necesarias = [
    "country",
    "category",
    "revenue"
]

def leer_todo_parquet_pandas():
    return pd.read_parquet(parquet_path)

def leer_columnas_parquet_pandas():
    return pd.read_parquet(
        parquet_path,
        columns=columnas_necesarias
    )

t_all_cols, _ = medir(
    leer_todo_parquet_pandas,
    repeticiones=3
)

t_selected_cols, pd_selected = medir(
    leer_columnas_parquet_pandas,
    repeticiones=3
)

print(f"Todas las columnas:      {t_all_cols:.5f} s")
print(f"Solo columnas necesarias:{t_selected_cols:.5f} s")
print()
print(pd_selected.head())

### ¿Por qué puede funcionar tan bien?

Parquet almacena la información **por columnas**.

Si una consulta necesita únicamente 3 columnas de 50, un motor compatible puede evitar leer gran parte de la información restante.

En CSV, cada registro está representado como una línea de texto completa. Aunque existen técnicas para seleccionar columnas, el fichero debe ser interpretado desde texto y no ofrece las mismas ventajas estructurales de un formato columnar.

---

# 18. CSV también puede optimizarse

No debemos sacar la conclusión:

> “CSV es malo”.

CSV tiene ventajas importantes:

- extremadamente simple;
- legible por personas;
- compatible con prácticamente cualquier herramienta;
- excelente para intercambio de datos;
- fácil de generar.

Y Pandas permite optimizar su lectura.

Por ejemplo, usando `usecols`.

In [ ]:
def pandas_csv_usecols():
    return pd.read_csv(
        csv_path,
        usecols=[
            "country",
            "category",
            "revenue"
        ]
    )

t_csv_usecols, csv_selected = medir(
    pandas_csv_usecols,
    repeticiones=3
)

print(f"CSV completo con Pandas:       {t_pd_csv:.5f} s")
print(f"CSV solo columnas necesarias:  {t_csv_usecols:.5f} s")

También podríamos indicar tipos de datos explícitamente o procesar el CSV por fragmentos.

---

# 19. Cuando el CSV no cabe cómodamente en memoria: chunks

Pandas permite procesar un fichero CSV por bloques:

```python
chunksize=...
```

Esto es útil cuando queremos realizar una agregación sin cargar todo el archivo simultáneamente.

In [ ]:
ventas_por_pais = {}

for chunk in pd.read_csv(
    csv_path,
    usecols=["country", "revenue"],
    chunksize=100_000
):

    parcial = (
        chunk
        .groupby("country")["revenue"]
        .sum()
    )

    for country, value in parcial.items():
        ventas_por_pais[country] = (
            ventas_por_pais.get(country, 0)
            + value
        )

ventas_por_pais

Esto demuestra una idea importante:

> **Pandas no necesita cargar siempre todo el dataset a la vez.**

Aunque para cargas realmente grandes pueden resultar más adecuadas otras herramientas o arquitecturas, existen estrategias intermedias.

---

# 20. Polars Lazy + CSV

`scan_*` no está limitado a Parquet.

También podemos construir una consulta lazy sobre CSV:

In [ ]:
query_csv = (
    pl.scan_csv(csv_path)

    .filter(
        pl.col("country") == "ES"
    )

    .group_by("category")

    .agg(
        pl.col("revenue").sum().alias("revenue")
    )

    .sort(
        "revenue",
        descending=True
    )
)

query_csv.collect()

Pero Parquet sigue teniendo ventajas importantes:

- almacenamiento columnar;
- tipos;
- compresión;
- metadatos;
- lectura selectiva eficiente.

---

# 21. Construir nuestro resumen final

Vamos a reunir algunos resultados obtenidos en este entorno.

In [ ]:
resumen = pd.DataFrame({
    "Métrica": [
        "Tamaño CSV (MB)",
        "Tamaño Parquet (MB)",
        "Lectura Pandas CSV (s)",
        "Lectura Pandas Parquet (s)",
        "Lectura Polars CSV (s)",
        "Lectura Polars Parquet (s)",
        "GroupBy Pandas (s)",
        "GroupBy Polars (s)",
        "Consulta Polars Lazy (s)"
    ],
    "Resultado": [
        round(csv_mb, 2),
        round(parquet_mb, 2),
        round(t_pd_csv, 4),
        round(t_pd_parquet, 4),
        round(t_pl_csv, 4),
        round(t_pl_parquet, 4),
        round(t_group_pd, 5),
        round(t_group_pl, 5),
        round(t_lazy, 5)
    ]
})

resumen

# 22. Cuidado con los benchmarks

Un error común sería ejecutar este notebook y concluir:

> “Polars es X veces más rápido que Pandas.”

Eso sería demasiado simplista.

El rendimiento depende de:

- operación;
- tamaño;
- tipos de datos;
- versión de las librerías;
- número de núcleos;
- memoria;
- disco;
- caché;
- formato;
- optimizaciones;
- estructura concreta del problema.

Además, al repetir lecturas, el sistema operativo puede conservar información en caché.

---

### Lo que sí podemos concluir

Los experimentos permiten observar que:

1. **el formato importa**;
2. **el motor importa**;
3. **leer menos datos puede ser más importante que procesarlos más rápido**;
4. las optimizaciones automáticas pueden ahorrar trabajo;
5. no existe una herramienta universalmente mejor para todos los problemas.

---

# 23. ¿Cuándo utilizar cada uno?

## Pandas

Muy apropiado para:

- análisis exploratorio;
- notebooks;
- datasets pequeños y medianos;
- ecosistema Python de ciencia de datos;
- compatibilidad con muchas librerías;
- trabajo interactivo.

## Polars

Especialmente interesante cuando:

- trabajamos con datasets más grandes;
- realizamos muchas transformaciones;
- queremos aprovechar ejecución paralela;
- queremos utilizar consultas lazy;
- queremos optimizaciones del plan de ejecución.

## CSV

Útil cuando:

- necesitamos máxima compatibilidad;
- intercambiamos datos;
- queremos un formato sencillo;
- el volumen no es enorme;
- necesitamos inspeccionar manualmente el fichero.

## Parquet

Muy apropiado cuando:

- almacenamos datasets analíticos;
- tenemos muchas columnas;
- consultamos subconjuntos de columnas;
- queremos conservar tipos;
- queremos compresión;
- trabajamos con ecosistemas Big Data.

---

# 24. Mini-retos

## Reto A — Aumenta el volumen

Prueba:

```python
N_ROWS = 1_500_000
```

Regenera los archivos.

¿Las diferencias aumentan?

---

## Reto B — Más columnas

Añade varias columnas nuevas al dataset.

Después compara:

```python
pd.read_parquet(
    parquet_path,
    columns=["country", "category", "revenue"]
)
```

con la lectura completa.

¿Qué ocurre?

---

## Reto C — Más filtros

Construye una consulta Polars Lazy que seleccione:

- España;
- canal `app`;
- `revenue > 100`;
- categorías `electronics` y `home`.

Después utiliza:

```python
.explain(optimized=True)
```

¿Qué observa el optimizador?

---

## Reto D — CSV por chunks

Calcula la facturación por categoría procesando el CSV con:

```python
chunksize=50_000
```

Comprueba que obtienes aproximadamente el mismo resultado que al cargar todo el dataset.

---

# 25. Preguntas finales

Intenta responder sin mirar las diapositivas.

### 1
¿Por qué Parquet suele ocupar menos espacio que CSV?

### 2
¿Por qué Parquet puede leer únicamente determinadas columnas de forma eficiente?

### 3
¿Qué significa que Polars sea capaz de trabajar en modo lazy?

### 4
¿Qué diferencia conceptual existe entre:

```python
pl.read_parquet(...)
```

y:

```python
pl.scan_parquet(...)
```

### 5
¿Qué es projection pushdown?

### 6
¿Qué es predicate pushdown?

### 7
¿Que Polars haya sido más rápido en este notebook significa que Pandas sea una mala herramienta?

### 8
¿En qué situación utilizarías CSV aunque Parquet fuese más eficiente?

---

# 26. Idea final

En Big Data no basta con preguntar:

> **¿Qué código funciona?**

También debemos preguntar:

> **¿Cuántos datos estoy leyendo?**  
> **¿Cómo están almacenados?**  
> **¿Necesito todas las columnas?**  
> **¿Puedo filtrar antes?**  
> **¿Qué herramienta ejecutará mejor este patrón de trabajo?**

---

## Conexión con el siguiente notebook

Hasta ahora hemos trabajado principalmente con datos almacenados.

El siguiente reto será distinto:

> **¿qué ocurre cuando los datos no están esperando en un fichero, sino que llegan continuamente?**

### Notebook 3
**Streaming simulado · eventos · ventanas · latencia · datos tardíos**